In [6]:
import pandas as pd
import json
import os

def excel_to_json(input_file, output_file):
    # Read the Excel file
    excel_file = pd.ExcelFile(input_file)
    all_data = []

    # Iterate through each sheet in the Excel file
    for sheet_index, sheet_name in enumerate(excel_file.sheet_names):
        df = pd.read_excel(excel_file, sheet_name=sheet_name)
        sheet_suffix = f"{sheet_index + 1:02d}"  # Ensure unique IDs for each tab
        
        # Create the first icon element from the first row of data
        first_row = df.iloc[0]
        icon_element = {
            "type": "icn",
            "role": "image",
            "alt": first_row.get("alt", ""),
            "id": f"icn{sheet_suffix}",
            "content": first_row.get("content", ""),
            "left": first_row.get("left", ""),
            "top": first_row.get("top", ""),
            "action": "openGroupGhost",
            "target": f"grp{sheet_suffix}"
        }
        all_data.append(icon_element)
        
        # Build the group object
        group = {
            "type": "grp",
            "id": f"grp{sheet_suffix}",
            "style": "grpAdv",
            "left": "4em",
            "top": "2em",
            "width": "56em",
            "height": "25em",
            "visible": "false",
            "children": []
        }
        
        # Iterate through rows to create child objects
        for index, row in df.iterrows():
            if index == 0:
                continue  # Skip the first row which has already been processed as an icon element
            
            # Create the child object by omitting blank fields
            child = {key: value for key, value in row.items() if pd.notna(value) and value != ""}
            
            if child.get('style') == 'words':
                child['children'] = [
                    {
                        "type": "icn",
                        "role": "image",
                        "content": child.get('content'),
                        "left": "-4em",
                        "top": "-1em",
                        "height": "8em",
                        "width": "8em"
                    },
                    {
                        "type": "txt",
                        "style": "vocab",
                        "content": child.get('text'),
                        "left": "-.0em",
                        "top": ".25em"
                    }
                ]
            
            group['children'].append(child)
        
        all_data.append(group)
    
    # Write to JSON file without square brackets
    with open(output_file, 'w') as json_file:
        for i, item in enumerate(all_data):
            json.dump(item, json_file, indent=4)
            if i < len(all_data) - 1:
                json_file.write(",\n")

# Specify the input and output file paths for each Excel file
file_mappings = {
    "datavv.xlsx": "output1.json",
    "datavv2.xlsx": "output2.json",
    "datacc.xlsx": "output3.json",
    "datacc2.xlsx": "output4.json"
}

# Process each file
for input_file, output_file in file_mappings.items():
    excel_to_json(input_file, output_file)


Now we will insert the json snippets into the framework file.

In [7]:
def replace_placeholders(shell_file, output_files, output_file):
    with open(shell_file, 'r') as file:
        shell_data = file.read()

    # Read the contents of the JSON snippets as text
    with open(output_files[0], 'r') as file1, \
         open(output_files[1], 'r') as file2, \
         open(output_files[2], 'r') as file3, \
         open(output_files[3], 'r') as file4:
        
        output1 = file1.read()
        output2 = file2.read()
        output3 = file3.read()
        output4 = file4.read()
    
    # Replace placeholders with the actual content from the JSON files
    shell_data = shell_data.replace("/*ADD output1.json here*/", output1)
    shell_data = shell_data.replace("/*ADD output2.json here*/", output2)
    shell_data = shell_data.replace("/*ADD output3.json here*/", output3)
    shell_data = shell_data.replace("/*ADD output4.json here*/", output4)

    # Write the final result to the output file
    with open(output_file, 'w') as file:
        file.write(shell_data)

# File paths
shell_file = '../data/soundwallShell.json'
output_files = ['output1.json', 'output2.json', 'output3.json', 'output4.json']
output_file = 'soundwall.json'

# Run the function
replace_placeholders(shell_file, output_files, output_file)
